In [128]:
!pip install --break-system-packages language-tool-python

In [129]:
!pip install language-tool-python

In [115]:
import re
import os
import json
from pathlib import Path
from difflib import SequenceMatcher
from typing import Dict, List, Tuple

In [116]:
# =============================================================================
# STEP 1: LOAD DICTIONARIES
# =============================================================================
# Load medications, abbreviations, and medical terms from files

def load_list(file_path: str) -> List[str]:
    """Load simple list from file (one item per line)"""
    items = []
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            items = [line.strip().lower() for line in f if line.strip()]
    except FileNotFoundError:
        print(f"Warning: {file_path} not found")
    return items

def load_abbreviations(file_path: str) -> Dict[str, str]:
    """
    Load abbreviations file format: full_term,abbr1,abbr2,...
    Returns dict: {abbr: full_term}
    """
    abbrev_dict = {}
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line or ',' not in line:
                    continue
                parts = [p.strip().lower() for p in line.split(',')]
                correct = parts[0]
                for variant in parts[1:]:
                    abbrev_dict[variant] = correct
    except FileNotFoundError:
        print(f"Warning: {file_path} not found")
    return abbrev_dict

In [117]:
# =============================================================================
# STEP 2: LOAD TRANSCRIPTIONS
# =============================================================================
# Extract transcriptions from poly_gt annotation files

def load_transcriptions(label_path: str, transcription_key: str) -> List[Dict]:
    """Load OCR transcriptions from poly_gt file"""
    transcriptions = []
    try:
        with open(label_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split('\t', 1)
                if len(parts) == 2:
                    image_path = parts[0]
                    try:
                        items = json.loads(parts[1])
                        for item in items:
                            if transcription_key in item:
                                item['image'] = image_path
                                transcriptions.append(item)
                    except json.JSONDecodeError:
                        pass
    except Exception as e:
        print(f"Error reading {label_path}: {e}")
    return transcriptions

In [118]:
# =============================================================================
# STEP 3: SPLIT MIXED WORDS
# =============================================================================
# Separate numbers from letters: 2ampoules -> 2 ampoules

def split_mixed_words(text: str) -> str:
    """
    Separate numbers and letters: 2ampoules -> 2 ampoules , 03mots -> 03 mots
    """
    # digit followed by letters
    text = re.sub(r'(\d+)([a-zàâäéèêëïîôöœçñ]+)', r'\1 \2', text, flags=re.IGNORECASE)
    # letters followed by digits
    text = re.sub(r'([a-zàâäéèêëïîôöœçñ]+)(\d+)', r'\1 \2', text, flags=re.IGNORECASE)
    return text

In [119]:
# =============================================================================
# STEP 4: NORMALIZE DATES
# =============================================================================
# Convert all dates to jj/mm/aaaa format

def normalize_dates(text: str) -> str:
    """
    Normalize dates to jj/mm/aaaa format
    Handles: jj/mm/aaaa, jj-mm-aaaa, jj.mm.aaaa, jj:mm:aaaa, jj'mm'aaaa Also handles: jj / mm / aaaa (with spaces)
    If year is 2-digit, prepend 20
    If day/month is 1-digit, prepend 0
    After each date, adds newline
    """
    pattern = r'(\d{1,2})\s*[\s/:\-.\'](\d{1,2})\s*[\s/:\-.\'](\d{2,4})'
    
    def replace_date(match):
        day = match.group(1).zfill(2)
        month = match.group(2).zfill(2)
        year = match.group(3)
        if len(year) == 2:
            year = '20' + year
        return f"{day}/{month}/{year}\n"
    
    return re.sub(pattern, replace_date, text)

In [120]:
# =============================================================================
# STEP 5: REPLACE ABBREVIATIONS
# =============================================================================
# Replace abbreviations with full terms: tbl -> trouble

def replace_abbreviations(text: str, abbrev_dict: Dict[str, str]) -> Tuple[str, set]:
    """
    Replace abbreviations with full terms (case-insensitive)
    """
    words = text.split()
    result = []
    corrected_indices = set()
    
    for i, word in enumerate(words):
        word_lower = word.lower()
        # Keep apostrophes AND slashes for abbreviations like s/trt
        word_clean = re.sub(r"[^\w'/]", '', word_lower)
        
        if word_clean in abbrev_dict:
            result.append(abbrev_dict[word_clean])
            corrected_indices.add(i)
        else:
            result.append(word)
    
    ########## return tuple with indices - NEW: Return both text and flags ##########
    return ' '.join(result), corrected_indices

In [121]:
# =============================================================================
# STEP 6: FUZZY MATCHING WITH MINIMUM DISTANCE
# =============================================================================
# Find closest medical term or medication using string similarity

def find_closest_match(word: str, reference_list: List[str], threshold: float = 0.75) -> Tuple[str, bool]:
    """
    Find closest matching word in reference list using SequenceMatcher
    Returns: (matched_word, is_corrected_bool)
    """
    word_lower = word.lower()
    
    # Exact match
    if word_lower in reference_list:
        return word_lower, True
    
    best_match = None
    best_ratio = threshold
    
    for ref_word in reference_list:
        ratio = SequenceMatcher(None, word_lower, ref_word).ratio()
        if ratio > best_ratio:
            best_ratio = ratio
            best_match = ref_word
    
    return (best_match, True) if best_match else (word, False)

def apply_fuzzy_matching(text: str, medications: List[str], terms: List[str], tests: List[str]) -> Tuple[str, set]:
    """
    Returns: (corrected_text, set of corrected_word_indices)
    """
    words = text.split()
    result = []
    corrected_indices = set()
    
    for i, word in enumerate(words):
        # Try medication match first
        med_match, is_med_corrected = find_closest_match(word, medications, threshold=0.75)
        if is_med_corrected:
            result.append(med_match)
            corrected_indices.add(i)
            continue
        
        # Then try medical term match
        term_match, is_term_corrected = find_closest_match(word, terms, threshold=0.75)
        if is_term_corrected:
            result.append(term_match)
            corrected_indices.add(i)
            continue
        
        test_match, is_test_corrected = find_closest_match(word, tests, threshold=0.75)
        if is_test_corrected:
            result.append(test_match)
            
            #Flag corrected word
            corrected_indices.add(i)
            continue
        
        result.append(word)
    
    return ' '.join(result), corrected_indices

In [149]:
# =============================================================================
# STEP 7: GRAMMAR AND SPELLING CORRECTION
# =============================================================================
# Use built-in French spell checker

def correct_french_spelling(text: str, skip_indices: set = None, reference_lists: List[List[str]] = None) -> str:
    """
    Correct French spelling, but skip words that were already corrected AND skip words that exist in reference lists (medical terms/meds/tests)
    
    skip_indices: set of word positions to skip
    eference_lists: [medications, terms, tests] - words to NOT correct -
    """
    if skip_indices is None:
        skip_indices = set()
    if reference_lists is None:
        reference_lists = []
    
    # Create set of reference words to skip
    reference_words = set()
    for ref_list in reference_lists:
        for word in ref_list:
            reference_words.add(word.lower())
    
    try:
        from spellchecker import SpellChecker
        spell = SpellChecker(language='fr')
        
        words = text.split()
        corrected = []
        
        for i, word in enumerate(words):
            # Skip already-corrected words
            if i in skip_indices:
                corrected.append(word)
                continue
            
            word_lower = word.lower()
            word_clean = re.sub(r'[^\w]', '', word_lower)
            
            # Skip if word is in reference lists
            if word_clean in reference_words:
                corrected.append(word)
                continue
            
            # Keep numbers and punctuation
            if not re.search(r'[a-zàâäéèêëïîôöœçñ]', word, re.IGNORECASE):
                corrected.append(word)
                continue
            
            if word_clean not in spell:
                closest = spell.correction(word_clean)
                if closest:
                    corrected.append(closest)
                else:
                    corrected.append(word)
            else:
                corrected.append(word)
        
        return ' '.join(corrected)
    except ImportError:
        print("Warning: pyspellchecker not installed. Skipping spell check.")
        return text  

In [150]:
# =============================================================================
# STEP 8: SORT BY COORDINATES AND GROUP INTO LINES
# =============================================================================

def get_centroid(points: List[List[int]]) -> Tuple[float, float]:
    """Calculate center point of polygon"""
    if not points:
        return (float('inf'), float('inf'))
    y_coords = [p[1] for p in points]
    x_coords = [p[0] for p in points]
    centroid_y = sum(y_coords) / len(y_coords)
    centroid_x = sum(x_coords) / len(x_coords)
    return (centroid_y, centroid_x)

def sort_by_coordinates(items: List[Dict]) -> List[Dict]:
    """Sort by Y then X coordinates (reading order)"""
    def sort_key(item):
        points = item.get('points', [])
        centroid_y, centroid_x = get_centroid(points)
        return (centroid_y, centroid_x)
    return sorted(items, key=sort_key)

def group_into_lines(items: List[Dict], y_threshold: int = 100) -> List[List[Dict]]:
    """Group items into lines based on Y coordinate discontinuity"""
    if not items:
        return []
    lines = [[items[0]]]
    for i in range(1, len(items)):
        current_y, _ = get_centroid(items[i].get('points', []))
        prev_y, _ = get_centroid(items[i-1].get('points', []))
        if abs(current_y - prev_y) > y_threshold:
            lines.append([items[i]])
        else:
            lines[-1].append(items[i])
    return lines


In [151]:
# =============================================================================
# STEP 9: CLEAN NOISE
# =============================================================================

def clean_text(text: str) -> str:
    """Normalize whitespace and trim"""
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def remove_noise(text: str, noise_patterns: List[str]) -> str:
    """
    Remove unwanted words or regex patterns from text
    noise_patterns: list of words or regex patterns to remove
    Example: ['MouMNi', r'#{2,}', '###']
    """
    for pattern in noise_patterns:
        try:
            # Try as regex first
            text = re.sub(pattern, '', text, flags=re.IGNORECASE)
        except re.error:
            # If not valid regex, treat as literal string
            text = text.replace(pattern, '')
    
    # Clean up extra whitespace
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [155]:
# =============================================================================
# MAIN PREPROCESSING PIPELINE
# =============================================================================


def postprocess_ocr(
    transcriptions: List[Dict],
    abbrev_dict: Dict[str, str],
    medications: List[str],
    terms: List[str],
    tests: List[str],
    noise_patterns: List[str] = None
) -> str:
    """
    Complete OCR post-processing pipeline:
    1. Extract transcription text
    2. Split mixed words
    3. Normalize dates
    4. Replace abbreviations
    5. Apply fuzzy matching
    6. Correct spelling
    7. Remove noise
    8. Sort spatially and group into lines
    9. Join and clean
    """
    if noise_patterns is None:
        noise_patterns = ['MouMNi', 'moumni', 'Moumni']
    
    # Sort by coordinates
    transcriptions = sort_by_coordinates(transcriptions)
    
    # Group by image
    image_groups = {}
    for item in transcriptions:
        image = item['image']
        if image not in image_groups:
            image_groups[image] = []
        image_groups[image].append(item)
    
    # Process each image
    final_parts = []
    for image in sorted(image_groups.keys()):
        items = image_groups[image]
        lines = group_into_lines(items, y_threshold=50)
        
        line_texts = []
        for line_items in lines:
            line_items = sorted(line_items, key=lambda item: get_centroid(item.get('points', []))[1])
            line_words = []
            for item in line_items:
                text = item['transcription']
                if re.match(r'^\s*(MouMNi|Moumni|moumni|psychologue)\s*$', text, re.IGNORECASE):
                    continue
                all_corrected_indices = set()
                
                # Apply all postprocessing steps
                text = split_mixed_words(text)
                text = normalize_dates(text)
                
                # Replace abbreviations and flag them
                text, abbrev_indices = replace_abbreviations(text, abbrev_dict)
                all_corrected_indices.update(abbrev_indices)
                
                # Apply fuzzy matching and flag them (medications, terms, tests)
                text, fuzzy_indices = apply_fuzzy_matching(text, medications, terms, tests)
                all_corrected_indices.update(fuzzy_indices)
                
                # Correct spelling but skip flagged words and words in reference lists
                text = correct_french_spelling(text, reference_lists=[medications, tests, ['CM','TM','ro']])
                
                # Remove noise and clean
                text = remove_noise(text, noise_patterns)
                text = clean_text(text)
                
                if text:
                    line_words.append(text)
            
            if line_words:
                line_text = ' '.join(line_words)
                line_texts.append(line_text)
        
        image_text = '\n'.join(line_texts)
        final_parts.append(f"{image_text}")
    
    return '\n\n'.join(final_parts)

In [156]:
# =============================================================================
# PROCESS ALL FILES
# =============================================================================

def postprocess_all_ocr(
    transcription_key:str,
    base_path: str,
    annotations_folders: List[str],
    annotations_prefix: str,
    output_base_path: str,
    abbrev_path: str,
    medications_path: str,
    terms_path: str,
    tests_path: str,
    noise_patterns: List[str] = None
):
    """
        Post-process all OCR annotation files from both years (2016, 2021)
        Maintains folder structure: output/annotations_year/class/id/files
        
        noise_patterns: list of words/regex to remove
    """
    
    if noise_patterns is None:
        noise_patterns = ['MouMNi']
    
    # Load dictionaries
    abbrev_dict = load_abbreviations(abbrev_path)
    medications = load_list(medications_path)
    terms = load_list(terms_path)
    tests = load_list(tests_path)
    
    print(f"Loaded {len(abbrev_dict)} abbreviations")
    print(f"Loaded {len(medications)} medications")
    print(f"Loaded {len(terms)} medical terms")
    print(f"Loaded {len(tests)} tests\n")
    
    os.makedirs(output_base_path, exist_ok=True)
    
    # Process both years
    for year_folder in annotations_folders:
        year_path = os.path.join(base_path, year_folder)
        if not os.path.exists(year_path):
            print(f"Skipping {year_path} - not found")
            continue
        
        # Process each category
        for category in os.listdir(year_path):
            category_path = os.path.join(year_path, category)
            if not os.path.isdir(category_path):
                continue
            
            # Process each ID folder
            for id_folder in os.listdir(category_path):
                id_path = os.path.join(category_path, id_folder)
                if not os.path.isdir(id_path):
                    continue
                
                output_id_path = os.path.join(output_base_path, year_folder, category, id_folder)
                os.makedirs(output_id_path, exist_ok=True)
                
                # Process each poly_gt file
                for file in sorted(os.listdir(id_path)):
                    if file.startswith(annotations_prefix) and file.endswith('.txt'):
                        label_path = os.path.join(id_path, file)
                        output_file = os.path.join(output_id_path, file)
                        
                        try:
                            # Load and post-process
                            transcriptions = load_transcriptions(label_path,transcription_key)
                            result = postprocess_ocr(
                                transcriptions,
                                abbrev_dict,
                                medications,
                                terms,
                                tests,
                                noise_patterns
                            )
                            
                            # Extract metadata
                            year = year_folder.split('_')[-1]
                            metadata = f"ANNEE DE LA PREMIERE CONSULTATION: {year}\nIDENTIFIANT DU PATIENT: {id_folder}\nDIAGNOSTIQUE: {category}\n\n"
                            final_text = metadata + result
                            
                            # Save
                            with open(output_file, 'w', encoding='utf-8') as f:
                                f.write(final_text)
                            
                            print(f"✓ {year_folder}/{category}/{id_folder}/{file}")
                        except Exception as e:
                            print(f"✗ Error: {label_path} - {e}")

In [157]:
# =============================================================================
# RUN
# =============================================================================

if __name__ == "__main__":

    transcription_key='transcription'
    base_path = './data'
    annotations_folders = ['annotated_2016', 'annotated_2021']
    annotation_prefix = 'poly_gt_'
    output_base_path = './postprocessed_output'
    abbrev_path = 'dictionnaries/abreviations.txt'
    medications_path = 'dictionnaries/medications.txt'
    tests_path = 'dictionnaries/tests.txt'
    terms_path = 'dictionnaries/terms.txt'
    noise_patterns = ['MouMNi','Moumni','moumni']
    
    postprocess_all_ocr(
        transcription_key,
        base_path,
        annotations_folders,
        annotation_prefix,
        output_base_path,
        abbrev_path,
        medications_path,
        terms_path,
        tests_path,
        noise_patterns
    )
    
    print(f"\n✓ Complete! Output saved to: {output_base_path}")

Loaded 147 abbreviations
Loaded 34 medications
Loaded 138 medical terms
Loaded 7 tests

✓ annotated_2016/Neg/014/poly_gt_2016-014-01.txt
✓ annotated_2016/Neg/014/poly_gt_2016-014-02.txt
✓ annotated_2016/MA/051/poly_gt_2016-051-01.txt
✓ annotated_2016/MA/051/poly_gt_2016-051-02.txt
✓ annotated_2016/MA/051/poly_gt_2016-051-03.txt
✓ annotated_2016/MA/051/poly_gt_2016-051-04.txt
✓ annotated_2016/MA/051/poly_gt_2016-051-05.txt
✓ annotated_2016/MA/010/poly_gt_2016-010-01.txt
✓ annotated_2016/MA/010/poly_gt_2016-010-02.txt
✓ annotated_2016/MA/010/poly_gt_2016-010-03.txt
✓ annotated_2016/MA/010/poly_gt_2016-010-04.txt
✓ annotated_2016/MA/002/poly_gt_2016-002-01.txt
✓ annotated_2016/MA/002/poly_gt_2016-002-02.txt
✓ annotated_2016/MA/002/poly_gt_2016-002-03.txt
✓ annotated_2016/MA/002/poly_gt_2016-002-04.txt
✓ annotated_2016/MA/16_03390/poly_gt_2016-16_03390-01.txt
✓ annotated_2016/MA/16_03390/poly_gt_2016-16_03390-02.txt
✓ annotated_2016/MA/050/poly_gt_2016-050-01.txt
✓ annotated_2016/MA/050/po